In [ ]:
import pandas as pd

from parquet_files import COMPOSITE_DATA

composite_data = pd.read_parquet(COMPOSITE_DATA).dropna(subset=['p-7', 'p-6', 'p-5', 'p-4', 'p-3', 'p-2', 'p-1', 'p+0']).reset_index(drop=True)

In [20]:
composite_data['r1'] = (composite_data['p+1'] - composite_data['p+0']) / composite_data['p+0']
composite_data['r2'] = (composite_data['p+2'] - composite_data['p+0']) / composite_data['p+0']
composite_data['r3'] = (composite_data['p+3'] - composite_data['p+0']) / composite_data['p+0']
composite_data['r4'] = (composite_data['p+4'] - composite_data['p+0']) / composite_data['p+0']
composite_data['r5'] = (composite_data['p+5'] - composite_data['p+0']) / composite_data['p+0']
composite_data['r6'] = (composite_data['p+6'] - composite_data['p+0']) / composite_data['p+0']
composite_data['r7'] = (composite_data['p+7'] - composite_data['p+0']) / composite_data['p+0']

In [21]:
r1_data = composite_data.dropna(subset=['r1']).sort_values('date').reset_index(drop=True)
r2_data = composite_data.dropna(subset=['r2']).sort_values('date').reset_index(drop=True)
r3_data = composite_data.dropna(subset=['r3']).sort_values('date').reset_index(drop=True)
r4_data = composite_data.dropna(subset=['r4']).sort_values('date').reset_index(drop=True)
r5_data = composite_data.dropna(subset=['r5']).sort_values('date').reset_index(drop=True)
r6_data = composite_data.dropna(subset=['r6']).sort_values('date').reset_index(drop=True)
r7_data = composite_data.dropna(subset=['r7']).sort_values('date').reset_index(drop=True)

In [22]:
import torch.nn as nn
from torch.nn.utils.parametrizations import weight_norm
import torch
import tqdm

ch_fmt = [16, 32, 32]

class SimpleTCN(nn.Module):
  def __init__(self, kernel_size=2, dropout=0.2):
    super().__init__()
    layers = []
    for i in range(len(ch_fmt)):
      in_ch = 3 if i == 0 else ch_fmt[i-1]
      out_ch = ch_fmt[i]
      layers += [
        weight_norm(nn.Conv1d(in_ch, out_ch, kernel_size, padding=(kernel_size-1))),
        nn.ReLU(),
        nn.Dropout(dropout),
      ]
    self.network = nn.Sequential(*layers)
    self.fc = nn.Linear(ch_fmt[-1], 1)

  def forward(self, x):
    y = self.network(x.transpose(1, 2))
    y = y[:, :, -1]
    out = self.fc(y)
    return out.squeeze(1)

In [23]:

def train_tcn(train_loader, val_loader):
  tcn = SimpleTCN()
  optimizer = torch.optim.Adam(tcn.parameters(), lr=1e-3)
  criterion = nn.MSELoss()

  scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, 
    mode='min', 
    factor=0.5, 
    patience=3, 
  )

  early_stop_patience = 7
  best_val_loss = float('inf')
  epochs_no_improve = 0
  epochs = 50

  for _ in tqdm.tqdm(range(epochs)):
    tcn.train()
    train_losses = []

    for xb, yb in train_loader:
      optimizer.zero_grad()
      pred = tcn(xb)
      loss = criterion(pred, yb)
      loss.backward()
      optimizer.step()
      train_losses.append(loss.item())

    tcn.eval()
    val_losses = []

    with torch.no_grad():
      for xb, yb in val_loader:
        pred = tcn(xb)
        loss = criterion(pred, yb)
        val_losses.append(loss.item())

    val_loss = sum(val_losses) / len(val_losses)
    scheduler.step(val_loss)

    if val_loss < best_val_loss:
      best_val_loss = val_loss
      epochs_no_improve = 0
      best_weights = tcn.state_dict()
    else:
      epochs_no_improve += 1
      if epochs_no_improve >= early_stop_patience:
        break

  tcn.load_state_dict(best_weights)
  return tcn, best_val_loss ** 0.5

In [24]:
import lightgbm as lgbm
from sklearn.metrics import root_mean_squared_error

def train_lightgbm_model(X_train, Y_train, X_val, Y_val):
    train_set = lgbm.Dataset(X_train, label=Y_train)
    val_set = lgbm.Dataset(X_val, label=Y_val)

    params = {
        'objective': 'regression',
        'metric': 'rmse',
        'boosting_type': 'gbdt',
        'num_leaves': 31,
        'learning_rate': 0.03,
        'feature_fraction': 0.9,
        'bagging_fraction': 0.8,
        'bagging_freq': 5,
        'min_data_in_leaf': 50,
        'lambda_l1': 0.1,
        'lambda_l2': 0.1,
        'verbose': -1
    }

    model = lgbm.train(
        params,
        train_set,
        num_boost_round=2000,
        valid_sets=[train_set, val_set],
        valid_names=['train','val'],
    )

    preds = model.predict(X_val, num_iteration=model.best_iteration)
    rmse = root_mean_squared_error(Y_val, preds)

    return model, rmse

In [25]:
import numpy as np
from sklearn.linear_model import Ridge
from xgboost import XGBRegressor
from torch.utils.data import DataLoader, TensorDataset

price_cols = ['p-7', 'p-6', 'p-5', 'p-4', 'p-3', 'p-2', 'p-1', 'p+0']
sentiment_cols = ['custom_sentiment', 'finbert_sentiment']
input_features = price_cols + sentiment_cols

seq_length = len(price_cols)

def build_sequences(df, target_col):
  X = []
  y = []
  for _, row in df.iterrows():
    seq_prices = row[price_cols].values.reshape(-1, 1)
    seq_sentiment = np.tile(row[sentiment_cols].values, (seq_length, 1))
    seq = np.hstack([seq_prices, seq_sentiment])
    X.append(seq)
    y.append(row[target_col])
  return np.array(X, dtype=np.float32), np.array(y, dtype=np.float32)

best_rmse = np.inf
best_model = None
best_config = None
perfs = []

for df, target in [(r1_data, 'r1'), (r2_data, 'r2'), (r3_data, 'r3'), (r4_data, 'r4'), (r5_data, 'r5'), (r6_data, 'r6'), (r7_data, 'r7')]:
  for ignore_settings in [[], ['on_volatile_date'], ['on_volatile_date', 'adjacent_volatile_date'], ['on_volatile_date', 'adjacent_volatile_date', 'very_near_volatile_date'], ['on_volatile_date', 'adjacent_volatile_date', 'very_near_volatile_date', 'near_volatile_date']]:
    for to_ignore in ignore_settings:
      df = df[df[to_ignore] == False].reset_index(drop=True)
    if len(df) < 500:
      continue
    df = df.sort_values('date')
    split_idx = int(len(df) * 0.8)
    train = df.iloc[:split_idx]
    val = df.iloc[split_idx:]
    X_train = train[input_features]
    X_val = val[input_features]
    Y_train = train[target]
    Y_val = val[target]

    print('training ridge')
    # Ridge
    ridge_model = Ridge(alpha=1.0)
    ridge_model.fit(X_train, Y_train)
    pred_ridge = ridge_model.predict(X_val)
    rmse_ridge = root_mean_squared_error(Y_val, pred_ridge)
    perfs.append(("Ridge", ignore_settings, target, rmse_ridge))

    if rmse_ridge < best_rmse:
      best_rmse = rmse_ridge
      best_model = ridge_model
      best_config = ("Ridge", ignore_settings, target)

    print('training xgboost')
    # XGBoost
    xgb = XGBRegressor(
      n_estimators=300,
      max_depth=3,
      learning_rate=0.05,
      subsample=0.8,
      colsample_bytree=0.8,
      objective='reg:squarederror',
      tree_method='hist',
      random_state=42
    )
    xgb.fit(X_train, Y_train)
    pred_xgb = xgb.predict(X_val)
    rmse_xgb = root_mean_squared_error(Y_val, pred_xgb)
    perfs.append(("XGBoost", ignore_settings, target, rmse_xgb))

    if rmse_xgb < best_rmse:
      best_rmse = rmse_xgb
      best_model = xgb
      best_config = ("XGBoost", ignore_settings, target)
    
    print('training lgbm')
    # LightGBM
    gbm_model, rmse_gbm = train_lightgbm_model(X_train, Y_train, X_val, Y_val)

    perfs.append(("GBM", ignore_settings, target, rmse_gbm))

    if rmse_gbm < best_rmse:
      best_rmse = rmse_gbm
      best_model = gbm_model
      best_config = ("GBM", ignore_settings, target)
    
    # TCN sequences
    print('builing tcn sequences')
    X_seq_train, y_seq_train = build_sequences(train, target)
    X_seq_val, y_seq_val = build_sequences(val, target)

    train_dataset = TensorDataset(torch.tensor(X_seq_train, dtype=torch.float32), torch.tensor(y_seq_train, dtype=torch.float32))
    val_dataset = TensorDataset(torch.tensor(X_seq_val, dtype=torch.float32), torch.tensor(y_seq_val, dtype=torch.float32))
    train_loader = DataLoader(train_dataset, batch_size=32, shuffle=False)
    val_loader = DataLoader(val_dataset, batch_size=32)

    print('training tcn')
    # TCN
    tcn_model, rmse_tcn = train_tcn(train_loader, val_loader)

    perfs.append(("TCN", ignore_settings, target, rmse_tcn))

    if rmse_tcn < best_rmse:
      best_rmse = rmse_tcn
      best_model = tcn_model
      best_config = ("TCN", ignore_settings, target)

training ridge
training xgboost
training lgbm
builing tcn sequences
training tcn


 16%|█▌        | 8/50 [00:36<03:13,  4.60s/it]


training ridge
training xgboost
training lgbm
builing tcn sequences
training tcn


 14%|█▍        | 7/50 [00:31<03:12,  4.48s/it]


training ridge
training xgboost
training lgbm
builing tcn sequences
training tcn


...

 14%|█▍        | 7/50 [00:25<02:35,  3.63s/it]


training ridge
training xgboost
training lgbm
builing tcn sequences
training tcn


 14%|█▍        | 7/50 [00:21<02:10,  3.04s/it]


training ridge
training xgboost
training lgbm
builing tcn sequences
training tcn


 16%|█▌        | 8/50 [00:22<02:00,  2.86s/it]


In [27]:
print(perfs)
print(best_config)
print(best_rmse)

[('Ridge', [], 'r1', 0.09342805733045141), ('XGBoost', [], 'r1', 0.051005607032236826), ('GBM', [], 'r1', 0.05153211735837242), ('TCN', [], 'r1', 0.04777536758049245), ('Ridge', ['on_volatile_date'], 'r1', 0.09373769548222752), ('XGBoost', ['on_volatile_date'], 'r1', 0.05109017225647395), ('GBM', ['on_volatile_date'], 'r1', 0.05158497838514908), ('TCN', ['on_volatile_date'], 'r1', 0.055408037951520425), ('Ridge', ['on_volatile_date', 'adjacent_volatile_date'], 'r1', 0.09504131124159172), ('XGBoost', ['on_volatile_date', 'adjacent_volatile_date'], 'r1', 0.050974112669663924), ('GBM', ['on_volatile_date', 'adjacent_volatile_date'], 'r1', 0.052717470902597756), ('TCN', ['on_volatile_date', 'adjacent_volatile_date'], 'r1', 0.04762653230935516), ('Ridge', ['on_volatile_date', 'adjacent_volatile_date', 'very_near_volatile_date'], 'r1', 0.10019207439540306), ('XGBoost', ['on_volatile_date', 'adjacent_volatile_date', 'very_near_volatile_date'], 'r1', 0.053912784059265056), ('GBM', ['on_volatil

In [35]:

df = r1_data.copy()
df[df['on_volatile_date'] == False].reset_index(drop=True)
df[df['adjacent_volatile_date'] == False].reset_index(drop=True)
df = df.sort_values('date').reset_index(drop=True)
print(len(df))
split_idx = int(len(df) * 0.8)
val = df.iloc[split_idx:]
X_val = val[input_features]
Y_val = val['r1']
X_seq_val, y_seq_val = build_sequences(val, 'r1')
val_dataset = TensorDataset(torch.tensor(X_seq_val, dtype=torch.float32), torch.tensor(y_seq_val, dtype=torch.float32))
val_loader = DataLoader(val_dataset, batch_size=32)

24401


In [ ]:
best_model.eval()

all_preds = []
all_reals = []
with torch.no_grad():
  for xb, yb in val_loader:
    preds = best_model(xb).squeeze()
    all_preds.append(preds)
    all_reals.append(yb.squeeze())

all_preds = np.concatenate(all_preds)
all_reals = np.concatenate(all_reals)
random_preds = np.random.default_rng().choice([-1, 1], size=len(all_reals))

sign_match = np.sign(all_preds) == np.sign(all_reals)
sign_accuracy = sign_match.mean()
random_match = np.sign(all_preds) == np.sign(random_preds)
random_accuracy = random_match.mean()

print(f"Sign accuracy   : {sign_accuracy:.3f}") 

Sign accuracy: 0.471
Random accuracy: 0.502
